# Chapter 2

# LangChain examples with Ollama (gemma4:e2b)


## Minimal prompt execution

In [ ]:
!pip install langchain langchain_ollama


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma4:e2b")


In [ ]:
prompt_input = """Write a coincise message to remind users \
to be vigilant about phishing attacks."""
response = llm.invoke(prompt_input)

print(response)


In [ ]:
print(response.content)


# Running prompts with LangChain + Ollama


## Basic prompt with LangChain + Ollama


In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma4:e2b")


In [2]:
prompt_input = """Write a coincise message to remind users 
to be vigilant about phishing attacks."""

response = llm.invoke(prompt_input)
print(response.content)

Here are a few options, depending on the tone you want to convey:

**Option 1: Short and Direct (Best for quick alerts)**
> **Stay vigilant. Be wary of all suspicious emails and links.**

**Option 2: Action-Oriented (Focus on behavior)**
> **Be vigilant against phishing attacks. Verify before you click.**

**Option 3: Very Concise (For banners or social media)**
> **Phishing Alert: Stay safe and be vigilant.**

**Option 4: Slightly More Formal (Emphasizing caution)**
> **Always be vigilant. Protect yourself by carefully scrutinizing suspicious communications.**


# Prompt templates

## Prompt template - implementing it with a Python function

In [3]:
def generate_text_summary_prompt(text, num_words, tone):
    return f"You are an experienced copywriter. Write a {num_words} words summary the the following text, using a {tone} tone: {text}"

In [4]:
segovia_aqueduct_text = """The Aqueduct of Segovia (Spanish: 
Acueducto de Segovia) is a Roman aqueduct in Segovia, Spain. 
It was built around the first century AD to channel water from 
springs in the mountains 17 kilometres (11 mi) away to the 
city's fountains, public baths and private houses, and was in 
use until 1973. 
Its elevated section, with its complete arcade of 167 arches, 
is one of the best-preserved Roman aqueduct bridges and the 
foremost symbol of Segovia, as evidenced by its presence on the 
city's coat of arms. 
The Old Town of Segovia and the aqueduct, were declared a UNESCO 
World Heritage Site in 1985. As the aqueduct lacks a legible 
inscription (one was apparently located in the structure's attic, 
or top portion[citation needed]), the date of construction cannot be 
definitively determined. The general date of the Aqueduct's 
construction was long a mystery, although it was thought to have 
been during the 1st century AD, during the reigns of the Emperors 
Domitian, Nerva, and Trajan. At the end of the 20th century, 
Géza Alföldy deciphered the text on the dedication plaque by 
studying the anchors that held the now missing bronze letters 
in place. He determined that Emperor Domitian (AD 81–96) ordered 
its construction[1] and the year 98 AD was proposed as the most 
likely date of completion.[2] However, in 2016 archeological 
evidence was published which points to a slightly later date, 
after 112 AD, during the government of Trajan or in the 
beginning of the government of emperor Hadrian, 
from 117 AD."""

input_prompt = generate_text_summary_prompt(
    text=segovia_aqueduct_text, 
    num_words=20,
    tone="knowledgeable and engaging")

response = llm.invoke(input_prompt)
print(response.content)

This stunning Roman aqueduct in Segovia channels water and stands as a UNESCO marvel, its exact construction date remaining a fascinating archaeological mystery.


## Prompt template - using LangChain's ChatPromptTemplate with Ollama


In [5]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
"""You are an experienced copywriter. 
Write a {num_words} words summary the the following text, 
using a {tone} tone: {text}""")

In [6]:
prompt = prompt_template.format(
    text=segovia_aqueduct_text,
    num_words=20, 
    tone="knowledgeable and engaging")

In [7]:
response = llm.invoke(prompt)
print(response.content)

The magnificent Roman aqueduct in Segovia channels ancient water, standing as a UNESCO symbol of masterful Roman engineering and history.


# Few shot prompt with LangChain + Ollama


## Using standard prompt

In [8]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma4:e2b")


In [9]:
prompt_input = """Classify the following numbers as Abra, Kadabra or Abra Kadabra:

3, 4, 5, 7, 8, 10, 11, 13, 35

Examples: 
6 // not divisible by 5, not divisible by 7 // None
15 // divisible by 5, not divisible by 7 // Abra
12 // not divisible by 5, not divisible by 7 // None
21 // not divisible by 5, divisible by 7 // Kadabra
70 // divisible by 5, divisible by 7 // Abra Kadabra
"""

response = llm.invoke(prompt_input)
print(response.content)

Here is the classification of the numbers based on their divisibility by 5 and 7:

| Number | Divisible by 5? | Divisible by 7? | Classification |
| :----: | :-------------: | :-------------: | :------------: |
| **3** | No | No | None |
| **4** | No | No | None |
| **5** | Yes | No | Abra |
| **7** | No | Yes | Kadabra |
| **8** | No | No | None |
| **10** | Yes | No | Abra |
| **11** | No | No | None |
| **13** | No | No | None |
| **35** | Yes | Yes | Abra Kadabra |


## Using Langchain's FewShotPromptTemplate

In [10]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

examples = [
  {
      "number": 6,
      "reasoning": "not divisible by 5 nor by 7",
      "result": "None"
  },
  {
      "number": 15,
      "reasoning": "divisible by 5 but not by 7",
      "result": "Abra"
  },
  {
      "number": 12,
      "reasoning": "not divisible by 5 nor by 7",
      "result": "None"
  },
  {
      "number": 21,
      "reasoning": "divisible by 7 but not by 5",
      "result": "Kadabra"
  },
  {
      "number": 70,
      "reasoning": "divisible by 5 and by 7",
      "result": "Abra Kadabra"
  } ]

example_prompt = PromptTemplate(input_variables=["number", "reasoning", "result"], template="{number} \\ {reasoning} \\ {result}")
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Classify the following numbers as Abra, Kadabra or Abra Kadabra: {comma_delimited_input_numbers}",
    input_variables=["comma_delimited_input_numbers"]
)

prompt_input = few_shot_prompt.format(comma_delimited_input_numbers="3, 4, 5, 7, 8, 10, 11, 13, 35.")

response = llm.invoke(prompt_input)
print(response.content)

Based on the provided rules:
* **Abra:** Divisible by 5 but not by 7.
* **Kadabra:** Divisible by 7 but not by 5.
* **Abra Kadabra:** Divisible by both 5 and 7 (i.e., divisible by 35).
* **None:** Not divisible by 5 and not divisible by 7.

Here is the classification for the given numbers:

| Number | Divisible by 5? | Divisible by 7? | Classification |
| :---: | :---: | :---: | :---: |
| **3** | No | No | None |
| **4** | No | No | None |
| **5** | Yes | No | **Abra** |
| **7** | No | Yes | **Kadabra** |
| **8** | No | No | None |
| **10** | Yes | No | **Abra** |
| **11** | No | No | None |
| **13** | No | No | None |
| **35** | Yes | Yes | **Abra Kadabra** |
